# Relative Active Graph — App Launcher

**Run all cells top-to-bottom to start the full application stack.**

| Component | Description |
|-----------|-------------|
| FastAPI backend | Training orchestrator + WebSocket metrics stream (`/api/*`) |
| React dashboard | Interactive Train & Inference UI (served from `frontend/`) |
| Streamlit UI | Lightweight fallback UI (`app.py`) |

> **Colab**: a public ngrok URL is printed automatically.  
> **Local**: access `http://localhost:3000` (React) or `http://localhost:8000` (API).

In [ ]:
# ── 1. Clone repo (Colab only — skipped when already present) ─────────────────
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/Relative_Active_Graph') if IN_COLAB else Path('.').resolve()

if IN_COLAB and not REPO_DIR.exists():
    os.system('git clone https://github.com/Eupham/Relative_Active_Graph.git /content/Relative_Active_Graph')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'lcs' / 'induction'))
sys.path.insert(0, str(REPO_DIR / 'lcs' / 'training'))
print(f'Working directory: {REPO_DIR}')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
from pathlib import Path

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# lcs/requirements.txt uses space-separated packages on one line — parse manually
req_text = Path('lcs/requirements.txt').read_text()
lcs_pkgs = [tok for line in req_text.splitlines() for tok in line.split() if tok.strip()]
pip(*lcs_pkgs)

# Backend + tunnel deps
pip('fastapi', 'uvicorn[standard]', 'pydantic>=2', 'pyngrok')

# Node / npm check for React frontend
npm_ok = subprocess.run(['npm', '--version'], capture_output=True).returncode == 0
if npm_ok:
    print('npm found — React frontend will be available.')
else:
    print('npm not found — React frontend skipped; Streamlit UI will be used instead.')

In [ ]:
# ── 3. Build Rust engine (skip if binary already exists) ─────────────────────
import subprocess
from pathlib import Path

release_bin = Path('target/release/csrre')
debug_bin   = Path('target/debug/csrre')

if release_bin.exists():
    print(f'Release binary already present: {release_bin}')
elif debug_bin.exists():
    print(f'Debug binary already present: {debug_bin}')
else:
    cargo = subprocess.run(['cargo', '--version'], capture_output=True)
    if cargo.returncode == 0:
        print('Building Rust engine (release)…')
        result = subprocess.run(['cargo', 'build', '--release'], capture_output=True, text=True)
        if result.returncode == 0:
            print('Build succeeded.')
        else:
            print('Release build failed — trying debug…')
            subprocess.run(['cargo', 'build'], check=True)
    else:
        print('cargo not found — Rust engine unavailable. Training will be limited to Python induction only.')

In [ ]:
# ── 4. Install React frontend deps ───────────────────────────────────────────
import subprocess
from pathlib import Path

npm_ok = subprocess.run(['npm', '--version'], capture_output=True).returncode == 0

if npm_ok and Path('frontend/package.json').exists():
    node_modules = Path('frontend/node_modules')
    if not node_modules.exists():
        print('Installing React dependencies (this may take a minute)…')
        subprocess.check_call(['npm', 'install', '--prefix', 'frontend', '--silent'])
    else:
        print('node_modules already present — skipping npm install.')

In [ ]:
# ── 5. Launch everything ──────────────────────────────────────────────────────
import os, sys, time, threading, subprocess
from pathlib import Path

REPO_DIR = Path('.').resolve()
IN_COLAB = 'google.colab' in sys.modules
npm_ok   = subprocess.run(['npm', '--version'], capture_output=True).returncode == 0

env = os.environ.copy()
env['PYTHONPATH'] = (
    str(REPO_DIR / 'lcs' / 'training') + os.pathsep +
    str(REPO_DIR / 'lcs' / 'induction') + os.pathsep +
    env.get('PYTHONPATH', '')
)

procs = []

# ── FastAPI backend ──
backend_log = open('/tmp/backend.log', 'w')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.server:app',
     '--host', '0.0.0.0', '--port', '8000', '--reload'],
    cwd=str(REPO_DIR), env=env,
    stdout=backend_log, stderr=backend_log,
)
procs.append(('FastAPI backend', backend_proc, 8000))
print('FastAPI backend starting on port 8000…')

# ── React frontend OR Streamlit ──
if npm_ok and Path('frontend/node_modules').exists():
    fe_log = open('/tmp/frontend.log', 'w')
    fe_env = env.copy()
    fe_env['BROWSER'] = 'none'          # suppress auto-open in Colab
    fe_env['PORT']    = '3000'
    fe_proc = subprocess.Popen(
        ['npm', 'start', '--prefix', 'frontend'],
        cwd=str(REPO_DIR), env=fe_env,
        stdout=fe_log, stderr=fe_log,
    )
    procs.append(('React dashboard', fe_proc, 3000))
    print('React dashboard starting on port 3000…')
else:
    st_log = open('/tmp/streamlit.log', 'w')
    st_proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.port', '8501',
         '--server.headless', 'true'],
        cwd=str(REPO_DIR), env=env,
        stdout=st_log, stderr=st_log,
    )
    procs.append(('Streamlit UI', st_proc, 8501))
    print('Streamlit UI starting on port 8501…')

# Brief pause so processes can bind
time.sleep(3)

# ── Expose via ngrok (Colab) or print local URLs ──
if IN_COLAB:
    from pyngrok import ngrok
    tunnels = {}
    for name, _, port in procs:
        public = ngrok.connect(port)
        tunnels[name] = public.public_url
    print('\n═══════════════════════════════════════')
    print('  APP READY — PUBLIC URLS (Colab)')
    print('═══════════════════════════════════════')
    for name, url in tunnels.items():
        print(f'  {name:22s}  {url}')
    print('═══════════════════════════════════════')
else:
    print('\n═══════════════════════════════════════')
    print('  APP READY — LOCAL URLS')
    print('═══════════════════════════════════════')
    for name, _, port in procs:
        print(f'  {name:22s}  http://localhost:{port}')
    print('═══════════════════════════════════════')
    print('\nLogs: /tmp/backend.log  /tmp/frontend.log  /tmp/streamlit.log')

In [ ]:
# ── 6. (Optional) Stop all services ──────────────────────────────────────────
# Run this cell to cleanly shut down every process started above.
for name, proc, port in procs:
    proc.terminate()
    print(f'Stopped {name} (pid {proc.pid})')

if IN_COLAB:
    from pyngrok import ngrok
    ngrok.kill()
    print('ngrok tunnels closed.')